# 05 · Corpus del Guardian: raccolta, parsing e filtro ICE

Questo notebook costruisce il secondo corpus giornalistico del progetto, affiancato a quello del New York Times (notebook 01). La ragione è un limite del corpus NYT: l'Archive API fornisce per ogni articolo solo titolo, abstract e paragrafo d'apertura, e il testo integrale non è accessibile (paywall e termini d'uso). La Guardian Open Platform espone invece il campo `bodyText`, cioè il testo completo, e rende possibili analisi che sul NYT non si possono fare: topic modeling sul testo intero (notebook 07) e rete dei tag editoriali (notebook 08).

**Accesso.** Chiave *Developer*, gratuita per uso non commerciale: 1 richiesta al secondo, 500 al giorno, testo degli articoli incluso. La chiave si legge da `.env` (`GUARDIAN_API_KEY`); l'intervallo tra le richieste è configurabile con `GUARDIAN_RATE_LIMIT_SECONDS` (default 1,1 s, poco sopra il limite).

**Il problema della disambiguazione.** Sul NYT il filtro è tassonomico: un articolo riguarda ICE se la redazione gli ha assegnato l'organizzazione `Immigration and Customs Enforcement (US)` (notebook 01 §2). Sul Guardian questa strada è percorribile solo in parte, per due ragioni verificate sui dati:

1. il tag dedicato `us-news/ice-us-immigration-and-customs-enforcement` esiste, ma la redazione lo applica in modo sistematico solo dal 2025. Tra il 2003 e il 2024 copre da 0 a 26 articoli l'anno (tabella in §3), quindi un filtro sul solo tag svuoterebbe i periodi precedenti e annullerebbe il confronto diacronico;
2. per convenzione redazionale il Guardian scrive la sigla come "Ice", identica al sostantivo *ice* a inizio frase e nei titoli. Una ricerca testuale sulla sigla violerebbe il vincolo di disambiguazione del progetto.

Il filtro è quindi diviso in due domande, ciascuna con il proprio metodo e la propria validazione:

- **disambiguazione** (§1-§4): l'articolo nomina l'agenzia federale? Si risponde con una regola deterministica: tag ICE presente, oppure nome completo dell'agenzia nel `bodyText`;
- **rilevanza** (§5): l'articolo parla di ICE o la cita di passaggio? Si risponde con un classificatore addestrato su un gold set annotato a mano. Il riferimento sono solo articoli del Guardian, mai le keyword del NYT: usarle renderebbe circolare il confronto tra le due testate.

**Struttura e output.** §1 raccolta (raw mai filtrato) → §2 parsing e deduplicazione → §3 filtro a regola → §4 validazione su campione → §5 filtro di rilevanza. Ogni sezione legge il proprio input da disco, quindi il parsing e i filtri si rieseguono senza nuove chiamate all'API. Output del §1-§4: `data/raw/guardian/search/{query}/{anno}.json` (raw), `data/processed/guardian/articles_candidates.parquet` (tutti i candidati) e `articles_ice_rule.parquet` (articoli che superano la regola).

## 0. Setup

Percorsi, costanti e configurazione. Due costanti condizionano tutto il resto del notebook:

- `QUERIES` definisce due interrogazioni indipendenti dell'endpoint `/search`: la ricerca per frase `"Immigration and Customs Enforcement"` e il filtro per tag ICE. Nessuna delle due basta da sola: il tag manca quasi del tutto prima del 2025, mentre la frase perde gli articoli taggati che nel testo usano solo la sigla. L'insieme dei candidati è l'unione dei risultati delle due query.
- `PAGE_SIZE = 200` è il massimo ammesso dall'API e riduce il numero di chiamate (circa 60 per l'intero periodo 2003-2026).

La cella successiva carica la chiave e il rate limit da `.env`. Se la chiave manca il notebook si ferma subito, invece di fallire a metà della raccolta.

In [ ]:
import datetime
import json
import logging
import os
import re
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import requests
from dotenv import load_dotenv

%matplotlib inline

# Radice del progetto: funziona sia lanciando da notebooks/ sia dalla radice
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
# Raw: risposte dell'API così come arrivano, mai filtrate (solo l'anno in corso viene riscaricato)
RAW_DIR = BASE_DIR / "data" / "raw" / "guardian" / "search"
PROCESSED_DIR = BASE_DIR / "data" / "processed" / "guardian"
# Unità (query, anno) completate e fallite, per riprendere un fetch interrotto
CHECKPOINT_PATH = RAW_DIR.parent / "checkpoint.json"
# Output del §3: tutti i candidati con i flag, e il sottoinsieme che supera la regola
OUT_CANDIDATES = PROCESSED_DIR / "articles_candidates.parquet"
OUT_ICE_RULE = PROCESSED_DIR / "articles_ice_rule.parquet"
FIG_DIR = BASE_DIR / "docs" / "figures"  # figure per la tesi

# Endpoint /search della Content API; tag dedicato a ICE (sistematico solo dal 2025)
SEARCH_URL = "https://content.guardianapis.com/search"
ICE_TAG = "us-news/ice-us-immigration-and-customs-enforcement"
ICE_FULLNAME = "Immigration and Customs Enforcement"
START_YEAR = 2003
PAGE_SIZE = 200  # massimo consentito dall'API
# Tentativi per pagina; al tentativo k si attendono k × BACKOFF_BASE_SECONDS secondi
MAX_RETRIES = 5
BACKOFF_BASE_SECONDS = 30

# Due query indipendenti: il tag da solo perde quasi tutto il pre-2025,
# la frase da sola perde articoli taggati che usano solo "Ice".
QUERIES = {
    "phrase": {"q": f'"{ICE_FULLNAME}"'},
    "tag": {"tag": ICE_TAG},
}

# Log con timestamp di ogni fetch e scrittura, come traccia per la riproducibilità
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", force=True)
log = logging.getLogger("guardian_pipeline")
print(f"BASE_DIR: {BASE_DIR}")

In [ ]:
def load_config() -> tuple[str, float]:
    """Legge da .env la chiave API e la pausa tra due richieste.

    Si ferma subito se la chiave manca, invece di fallire a metà della raccolta.
    Restituisce (api_key, rate_limit_seconds); la pausa ha default 1,1 s
    (limite della chiave Developer: 1 richiesta al secondo).
    """
    load_dotenv(BASE_DIR / ".env")
    api_key = os.getenv("GUARDIAN_API_KEY")
    if not api_key:
        raise RuntimeError("GUARDIAN_API_KEY non trovata in .env")
    # Pausa tra due richieste: il limite Developer è 1 richiesta/s, 1,1 s lascia un margine
    rate_limit_seconds = float(os.getenv("GUARDIAN_RATE_LIMIT_SECONDS", "1.1"))
    return api_key, rate_limit_seconds

api_key, rate_limit_seconds = load_config()
print(f"GUARDIAN_API_KEY caricata: {'si' if api_key else 'no'}")
print(f"Rate limit configurato: {rate_limit_seconds}s tra le richieste")

## 1. Raccolta dei dati: Guardian Content API

**Unità di lavoro.** La raccolta procede per coppie (query, anno): 2 query × 24 anni = 48 unità. Con `page-size=200` un anno occupa da 1 a 6 pagine di risultati, per circa 60 chiamate in tutto, molto al di sotto del limite di 500 al giorno. Il checkpoint (`data/raw/guardian/checkpoint.json`) registra le unità completate e quelle fallite e viene salvato dopo ogni unità: una raccolta interrotta riprende dalla prima unità mancante.

**Raw.** Per ogni unità si salva la lista delle pagine di risposta così come arrivano dall'API, in `data/raw/guardian/search/{query}/{anno}.json`. In questa fase non si filtra nulla: anche i risultati che la regola di §3 scarterà restano su disco, così una modifica del filtro non richiede nuove chiamate.

**Anno in corso.** L'anno corrente non viene mai marcato come completato e a ogni esecuzione viene riscaricato: il file nuovo sostituisce il precedente e, salvo rimozioni dal sito, ne è un sovrainsieme, perché include gli articoli pubblicati nel frattempo. È l'unica eccezione alla regola del progetto per cui il raw non si sovrascrive; gli anni chiusi non vengono più toccati.

**Parametri della richiesta.** `show-fields=headline,standfirst,byline,bodyText,wordcount` (titolo, sottotitolo, firma, testo, lunghezza) e `show-tags=keyword` (tag editoriali). `order-by=oldest` fissa l'ordine dei risultati, così la paginazione non salta né ripete articoli tra una pagina e l'altra.

**Errori.** Ogni pagina viene ritentata fino a 5 volte, con attesa crescente (30 s × numero del tentativo), sugli errori di rete e sulle risposte HTTP diverse da 200. Un 401 o 403 indica una chiave non valida e interrompe subito la raccolta. Un'unità che esaurisce i tentativi viene registrata come fallita e ritentata all'esecuzione successiva.

In [ ]:
def unit_key(query: str, year: int) -> str:
    """Chiave testuale di un'unità di lavoro nel checkpoint, es. "phrase:2018"."""
    return f"{query}:{year}"

def load_checkpoint() -> dict:
    """Legge il checkpoint (unità completate e fallite); se non esiste, parte da uno vuoto."""
    if CHECKPOINT_PATH.exists():
        return json.loads(CHECKPOINT_PATH.read_text(encoding="utf-8"))
    return {"completed_units": [], "failed_units": [], "last_run": None}

def save_checkpoint(checkpoint: dict) -> None:
    """Scrive il checkpoint su disco con l'ora (UTC) dell'ultimo aggiornamento; chiamata dopo ogni unità."""
    checkpoint["last_run"] = datetime.datetime.now(datetime.timezone.utc).isoformat()
    CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
    CHECKPOINT_PATH.write_text(json.dumps(checkpoint, indent=2), encoding="utf-8")

def raw_json_path(query: str, year: int) -> Path:
    """Percorso del file raw di un'unità: data/raw/guardian/search/{query}/{anno}.json."""
    return RAW_DIR / query / f"{year}.json"

def fetch_page(session: requests.Session, api_key: str, params: dict) -> dict:
    """Scarica una pagina di risultati da /search e restituisce il blocco "response" del JSON.

    Su errori di rete o risposte diverse da 200 ritenta fino a MAX_RETRIES volte, attendendo
    BACKOFF_BASE_SECONDS × numero del tentativo. Su 401/403 (chiave non valida) si ferma subito.
    Se tutti i tentativi falliscono solleva RuntimeError, che run_fetch registra come unità fallita.
    """
    # Errori di rete e HTTP != 200: nuovo tentativo con attesa crescente
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = session.get(SEARCH_URL, params={**params, "api-key": api_key}, timeout=30)
        except requests.exceptions.RequestException as exc:
            log.warning("Errore di rete (tentativo %d/%d): %s", attempt, MAX_RETRIES, exc)
            time.sleep(BACKOFF_BASE_SECONDS * attempt)
            continue
        if response.status_code == 200:
            return response.json()["response"]
        # Chiave non valida o non autorizzata: inutile ritentare
        if response.status_code in (401, 403):
            raise RuntimeError(f"API key non valida o non autorizzata (HTTP {response.status_code})")
        log.warning("HTTP %d (tentativo %d/%d): %s", response.status_code, attempt, MAX_RETRIES, response.text[:200])
        time.sleep(BACKOFF_BASE_SECONDS * attempt)
    raise RuntimeError(f"Fetch fallito dopo {MAX_RETRIES} tentativi: {params}")

def fetch_unit(session, api_key, rate_limit_seconds, query: str, year: int) -> list[dict]:
    """Scarica tutte le pagine di un'unità (query, anno) e le restituisce come lista di risposte.

    Il numero di pagine si legge dalla prima risposta; tra una pagina e l'altra attende
    rate_limit_seconds. Le risposte non vengono filtrate né modificate.
    """
    # Parametri comuni a tutte le pagine dell'unità: un anno solare, 200 risultati per pagina, ordine stabile
    base = {
        **QUERIES[query],
        "from-date": f"{year}-01-01",
        "to-date": f"{year}-12-31",
        "page-size": PAGE_SIZE,
        "order-by": "oldest",
        # bodyText = testo completo dell'articolo, il motivo per cui si usa il Guardian
        "show-fields": "headline,standfirst,byline,bodyText,wordcount",
        # Tag editoriali, da cui il parsing ricava has_ice_tag
        "show-tags": "keyword",
    }
    pages, page, n_pages = [], 1, 1
    # Il numero di pagine si conosce solo dopo la prima risposta (campo 'pages')
    while page <= n_pages:
        resp = fetch_page(session, api_key, {**base, "page": page})
        pages.append(resp)
        n_pages = resp.get("pages", 0)
        page += 1
        time.sleep(rate_limit_seconds)
    return pages

def run_fetch(start_year: int = START_YEAR, end_year: int | None = None) -> None:
    """Raccolta completa: scarica tutte le unità (query, anno) non ancora completate.

    Per ogni unità scarica le pagine, salva il raw e aggiorna il checkpoint. Un'unità fallita viene
    registrata e ritentata al run successivo. L'anno in corso non viene mai segnato come completato,
    quindi a ogni esecuzione viene riscaricato.
    """
    api_key, rate_limit_seconds = load_config()
    end_year = end_year or datetime.date.today().year
    current_year = datetime.date.today().year
    checkpoint = load_checkpoint()
    completed, failed = set(checkpoint["completed_units"]), set(checkpoint["failed_units"])

    # Unità ancora da fare: tutte le coppie (query, anno) non segnate come completate nel checkpoint
    units = [(q, y) for q in QUERIES for y in range(start_year, end_year + 1) if unit_key(q, y) not in completed]
    if not units:
        log.info("Niente da scaricare: tutte le unità risultano già completate.")
        return
    log.info("Unità da scaricare: %d", len(units))
    session = requests.Session()

    for query, year in units:
        key = unit_key(query, year)
        try:
            pages = fetch_unit(session, api_key, rate_limit_seconds, query, year)
        except RuntimeError as exc:
            log.error("%s: %s", key, exc)
            failed.add(key)
            checkpoint["failed_units"] = sorted(failed)
            save_checkpoint(checkpoint)
            continue
        # Il raw si scrive prima di aggiornare il checkpoint: un'unità completata ha sempre il suo file
        path = raw_json_path(query, year)
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(json.dumps(pages, ensure_ascii=False), encoding="utf-8")
        n = sum(len(p.get("results", [])) for p in pages)
        log.info("Salvato %s -> %s (%d articoli, %d pagine)", key, path.relative_to(BASE_DIR), n, len(pages))

        failed.discard(key)
        # L'anno in corso non si segna mai completato: viene riscaricato al run successivo
        if year < current_year:
            completed.add(key)
        checkpoint["completed_units"] = sorted(completed)
        checkpoint["failed_units"] = sorted(failed)
        save_checkpoint(checkpoint)

    log.info("Fatto. Completate: %d, fallite: %d", len(completed), len(failed))

In [ ]:
run_fetch()

## 2. Parsing e rimozione duplicati 

Il parsing legge solo i file raw e produce una tabella con una riga per articolo.

**Rimozione duplicati.** Un articolo taggato che contiene anche il nome completo compare nei risultati di entrambe le query. Le righe vengono unite sull'`id` Guardian (il percorso dell'articolo, es. `us-news/2018/jun/...`); le colonne `via_phrase` e `via_tag` registrano da quale query proviene ogni articolo.

**Campi estratti.** Dal JSON: data di pubblicazione, sezione, titolo, sottotitolo (`standfirst`), firma, testo, numero di parole, URL e lista dei tag `keyword`, da cui si ricava `has_ice_tag`.

**Conteggi calcolati sul `bodyText`.**

- `n_fullname`: occorrenze del nome completo, con la regex `\bImmigrations?\s+and\s+Customs\s+Enforcement\b` e senza distinzione tra maiuscole e minuscole. La prima versione cercava la stringa esatta e scartava articoli che nominano l'agenzia in forme come "US immigration and customs enforcement (Ice)" o con il refuso "Immigrations and Customs Enforcement": 5 dei 10 articoli scartati nel campione di §4 alla prima esecuzione.
- `n_acronym`: occorrenze di `ICE` o `Ice` come parola intera, con distinzione tra maiuscole e minuscole (il sostantivo minuscolo *ice* è escluso). Questo conteggio non entra mai nel criterio di inclusione, perché da solo non disambigua: misura soltanto quanto spazio l'agenzia occupa in articoli già selezionati dalla regola di §3. Dentro quegli articoli un "Ice" con un altro significato (a inizio frase, in un nome proprio) resta possibile ed è un rumore accettato per una misura di densità.

**Limite noto, correzione rinviata.** Il `bodyText` di 81 candidati (76 del 2026) contiene caratteri Unicode di formattazione invisibili (U+2060 *word joiner*, U+200C *zero-width non-joiner*) inseriti dentro le parole. Quando cadono nel nome dell'agenzia la regex non lo riconosce: 3 articoli vengono scartati a torto dal §3 e alcuni conteggi delle menzioni risultano sottostimati. La correzione (rimuovere questi caratteri prima del match) è nel backlog: cambierà di poco i numeri di questo notebook e dei notebook 07, 08 e 10.

In [ ]:
# Case-insensitive e plurale tollerato: il Guardian scrive anche "US immigration and customs
# enforcement (Ice)" e "Immigrations and Customs Enforcement" (trovati nella validazione §4).
FULLNAME_RE = re.compile(r"\bImmigrations?\s+and\s+Customs\s+Enforcement\b", re.IGNORECASE)
ACRONYM_RE = re.compile(r"\b(?:ICE|Ice)\b")

def extract_article(doc: dict, query: str) -> dict:
    """Trasforma un risultato dell'API in una riga della tabella degli articoli.

    Copia metadati e campi testuali, ricava has_ice_tag dall'elenco dei tag e calcola sul testo
    completo i due conteggi: n_fullname (nome completo, regex tollerante) e n_acronym (ICE/Ice).
    `query` registra da quale delle due interrogazioni proviene la riga.
    """
    fields = doc.get("fields") or {}
    tags = [t["id"] for t in doc.get("tags") or []]
    body = fields.get("bodyText") or ""
    return {
        "id": doc["id"],
        "pub_date": doc.get("webPublicationDate"),
        "section_id": doc.get("sectionId"),
        "section_name": doc.get("sectionName"),
        "pillar": doc.get("pillarName"),
        "headline": fields.get("headline") or doc.get("webTitle"),
        "standfirst": fields.get("standfirst"),
        "byline": fields.get("byline"),
        "body_text": body,
        "word_count": int(fields["wordcount"]) if fields.get("wordcount") else None,
        "web_url": doc.get("webUrl"),
        # Elenco completo dei tag, conservato come stringa JSON
        "keyword_tags_json": json.dumps(tags),
        "has_ice_tag": ICE_TAG in tags,
        "n_fullname": len(FULLNAME_RE.findall(body)),
        "n_acronym": len(ACRONYM_RE.findall(body)),
        # Query di provenienza: serve a costruire via_phrase / via_tag dopo la rimozione dei duplicati
        "query": query,
    }

def iter_raw_files():
    """Elenca in ordine tutti i file raw, per entrambe le query e tutti gli anni."""
    return sorted(RAW_DIR.glob("*/*.json"))

rows = []
# Ogni file raw è una lista di pagine; gli articoli sono nel campo 'results' di ciascuna
for path in iter_raw_files():
    query = path.parent.name
    for page in json.loads(path.read_text(encoding="utf-8")):
        rows.extend(extract_article(doc, query) for doc in page.get("results", []))

raw_df = pd.DataFrame(rows)
# Per ogni id, le query che lo hanno restituito (prima di eliminare i duplicati)
via = raw_df.groupby("id")["query"].agg(lambda s: sorted(set(s)))
# Rimozione duplicati: un articolo restituito da entrambe le query resta una volta sola
df = raw_df.drop_duplicates("id").drop(columns="query").set_index("id")
df["via_phrase"] = via.map(lambda q: "phrase" in q)
df["via_tag"] = via.map(lambda q: "tag" in q)
df = df.reset_index()
# format ISO8601 esplicito: evita conversioni silenziose in NaT con formati misti (problema già visto sul NYT)
df["pub_date"] = pd.to_datetime(df["pub_date"], utc=True, format="ISO8601")
df["year"] = df["pub_date"].dt.year
df = df.sort_values("pub_date").reset_index(drop=True)

print(f"Righe raw: {len(raw_df):,}  ->  articoli unici: {len(df):,}  (duplicati tra query: {len(raw_df) - len(df):,})")
print(f"Periodo: {df['pub_date'].min().date()} -> {df['pub_date'].max().date()}")
print(f"body_text vuoto: {(df['body_text'].str.len() == 0).sum()}")

## 3. Filtro a regola (disambiguazione)

`is_ice_rule = has_ice_tag OR n_fullname >= 1`

La regola accetta un articolo se la redazione gli ha assegnato il tag ICE oppure se il testo contiene almeno una volta il nome completo dell'agenzia. Entrambe le condizioni identificano l'agenzia senza ambiguità: il tag per costruzione, il nome completo perché non ha altri referenti.

**Perché ricalcolare il match sul testo.** La ricerca per frase dell'API non garantisce che la stringa compaia nel testo: lavora su un indice e non su un confronto esatto, e restituisce anche articoli in cui il nome completo non c'è. In un test preliminare 4 risultati su 18 non contenevano la frase; sull'intero corpus i risultati della query per frase privi sia del nome completo sia del tag sono 21 su 2.975 (0,7%). Il criterio di inclusione è quindi il match locale, non la risposta dell'API.

**Cosa la regola non stabilisce.** La regola dice se l'agenzia è nominata, non se l'articolo ne parla. Per questo `agency_mentions = n_fullname + n_acronym` viene salvato come colonna ma non usato come soglia. Separare gli articoli su ICE dalle citazioni di passaggio è il compito del §5; una soglia fissa sulle menzioni sarebbe arbitraria, perché un articolo lungo su ICE può nominare l'agenzia poche volte e un live-blog lunghissimo può nominarla molte volte di sfuggita.

**Output.** In `data/processed/guardian/`: `articles_candidates.parquet` (tutti i 3.313 candidati, con i flag) e `articles_ice_rule.parquet` (i 3.292 articoli accettati).

In [ ]:
# Regola di disambiguazione (§3): tag ICE oppure almeno una volta il nome completo nel testo
df["is_ice_rule"] = df["has_ice_tag"] | (df["n_fullname"] >= 1)
# Intensità delle menzioni: non è un criterio di inclusione, serve al filtro di rilevanza (§5)
df["agency_mentions"] = df["n_fullname"] + df["n_acronym"]

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
df.to_parquet(OUT_CANDIDATES, index=False)
ice = df[df["is_ice_rule"]].reset_index(drop=True)
ice.to_parquet(OUT_ICE_RULE, index=False)

print(f"Candidati: {len(df):,}  -> is_ice_rule: {len(ice):,}  (scartati: {len(df) - len(ice):,})")
print(f"  di cui con tag ICE: {ice['has_ice_tag'].sum():,}   solo nome completo: {(~ice['has_ice_tag']).sum():,}")
print(f"Scritti: {OUT_CANDIDATES.relative_to(BASE_DIR)}, {OUT_ICE_RULE.relative_to(BASE_DIR)}")

In [ ]:
# Riepilogo per anno: candidati, accettati, con tag, scartati, menzioni negli accettati
yearly = df.groupby("year").agg(
    candidati=("id", "size"),
    ice_rule=("is_ice_rule", "sum"),
    con_tag=("has_ice_tag", "sum"),
)
yearly["scartati"] = yearly["candidati"] - yearly["ice_rule"]
mentions = ice.groupby("year")["agency_mentions"].describe()[["50%", "mean"]].rename(columns={"50%": "menzioni_mediana", "mean": "menzioni_media"})
yearly = yearly.join(mentions.round(1))
yearly

**Come leggere la tabella.** Una riga per anno. `candidati`: articoli restituiti dall'unione delle due query; `ice_rule`: articoli accettati dalla regola; `con_tag`: quanti degli articoli accettati hanno il tag ICE; `scartati`: candidati respinti; le ultime due colonne riportano mediana e media delle menzioni dell'agenzia negli articoli accettati.

Due fatti contano per il resto del progetto. Il tag ICE copre al massimo 26 articoli l'anno fino al 2024 e diventa frequente solo dal 2025 (382 articoli, poi 701 nel 2026): è la conferma numerica che un filtro sul solo tag non è utilizzabile. Il volume cresce di un ordine di grandezza nel 2025-2026 (1.115 e 1.161 articoli contro un massimo precedente di 187 nel 2019). Il 2004 manca perché nessuna delle due query restituisce articoli; il 2026 è parziale (fino al 2026-09-25).

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
yearly[["ice_rule", "con_tag"]].plot(kind="bar", ax=ax, color=["#4C72B0", "#DD8452"], width=0.8)
ax.set_title("Guardian: ICE articles per year (rule filter) and articles with the ICE tag")
ax.set_xlabel("")
ax.set_ylabel("articles")
ax.legend(["rule filter (ICE tag OR full name in text)", "with ICE tag"], frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
FIG_DIR.mkdir(parents=True, exist_ok=True)
# Figura per la tesi, salvata in docs/figures
fig.savefig(FIG_DIR / "guardian_rule_by_year.png", dpi=200, bbox_inches="tight")
plt.show()

**Come leggere il grafico.** Barre blu: articoli accettati dalla regola, per anno. Barre arancioni: quanti di questi hanno il tag ICE. Prima del 2025 la distanza tra le due serie è la parte di corpus che un filtro sul solo tag avrebbe perso, cioè quasi tutta. La forma della serie (crescita dal 2014, massimo nel 2018-2019, calo nel 2021-2024, salto nel 2025) ricalca a grandi linee la copertura del NYT (notebook 01 §5). Il confronto in valore assoluto tra le due testate richiede però il filtro di rilevanza del §5, perché qui sono contate anche le citazioni di passaggio.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
# Valori oltre 30 raccolti nell'ultima barra: pochi live-blog con moltissime menzioni schiaccerebbero il grafico
ice["agency_mentions"].clip(upper=30).plot(kind="hist", bins=30, ax=ax, color="#4C72B0")
ax.set_title("Agency mentions per article (values above 30 grouped in the last bar)")
ax.set_xlabel("mentions (full name + acronym)")
ax.set_ylabel("articles")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
fig.savefig(FIG_DIR / "guardian_mentions_hist.png", dpi=200, bbox_inches="tight")
plt.show()
print((ice["agency_mentions"] <= 2).mean().round(3), "quota di articoli con <= 2 menzioni (candidati 'di passaggio')")

**Come leggere l'istogramma.** Distribuzione del numero di menzioni dell'agenzia (nome completo più sigla) per articolo accettato; i valori sopra 30 sono raccolti nell'ultima barra. Il 33,9% degli articoli (1.115 su 3.292) ha al massimo 2 menzioni. Sono i candidati più probabili a una citazione di passaggio, ma non lo sono tutti, ed è per questo che il §5 non usa una soglia su questo valore.

## 4. Validazione su campione

Ogni filtro nuovo va verificato a mano prima di essere applicato all'intero dataset, come già fatto per il filtro NYT (notebook 01). I controlli sono due e rispondono a domande diverse.

- **Falsi positivi di disambiguazione.** 20 articoli accettati, 5 per ciascuno dei quattro periodi usati in tutto il progetto (2003-2016, 2017-2020, 2021-2024, 2025-presente, definiti nel notebook 02). Senza stratificazione il campione sarebbe quasi tutto 2025-2026, dove si concentra il 69% del corpus. Domanda: il riferimento è all'agenzia federale e non ad altro?
- **Articoli scartati.** 10 dei 21 candidati respinti. Domanda: sono davvero articoli senza nome completo e senza tag, oppure la regola perde articoli che nominano l'agenzia?

Per ogni articolo la tabella mostra data, sezione, presenza del tag, numero di menzioni, titolo e il passaggio del testo attorno alla prima menzione (il nome completo se presente, altrimenti la sigla). Il seed è fissato (`seed=42`), quindi il campione è riproducibile.

In [ ]:
# I quattro periodi usati in tutto il progetto (notebook 02); 9999 = fino a oggi
PERIODS = [("2003-2016", 2003, 2016), ("2017-2020", 2017, 2020), ("2021-2024", 2021, 2024), ("2025-presente", 2025, 9999)]

def context_snippet(body: str, width: int = 110) -> str:
    """Restituisce il passaggio di testo attorno alla prima menzione dell'agenzia.

    Cerca prima il nome completo, poi la sigla, e prende `width` caratteri prima e dopo.
    Se non trova menzioni restituisce l'inizio del testo. Serve alla validazione manuale (§4).
    """
    m = FULLNAME_RE.search(body) or ACRONYM_RE.search(body)
    if not m:
        return body[:width * 2]
    return "…" + body[max(0, m.start() - width): m.end() + width].replace("\n", " ") + "…"

def show_sample(frame: pd.DataFrame, n: int, seed: int = 42) -> pd.DataFrame:
    """Estrae un campione riproducibile di n articoli e lo mostra come tabella leggibile.

    Colonne: data, sezione, tag, menzioni, titolo (troncato) e il passaggio di context_snippet,
    in ordine di data.
    """
    s = frame.sample(min(n, len(frame)), random_state=seed).sort_values("pub_date")
    return pd.DataFrame({
        "data": s["pub_date"].dt.date,
        "sezione": s["section_id"],
        "tag": s["has_ice_tag"],
        "menzioni": s["agency_mentions"],
        "titolo": s["headline"].str.slice(0, 90),
        "contesto": s["body_text"].map(context_snippet),
    })

pd.set_option("display.max_colwidth", 260)
samples = []
# 5 articoli per periodo: senza stratificazione il campione sarebbe quasi tutto 2025-2026
for label, y0, y1 in PERIODS:
    part = ice[(ice["year"] >= y0) & (ice["year"] <= y1)]
    samples.append(show_sample(part, 5).assign(periodo=label))
validation_sample = pd.concat(samples)
validation_sample

**Come leggere la tabella.** Un articolo per riga, ordinati per data entro ciascun periodo. La colonna `contesto` è il passaggio da cui si giudica la disambiguazione: in tutti i 20 casi il nome completo o la sigla si riferiscono all'agenzia federale. Le colonne `titolo` e `menzioni` servono invece al giudizio di rilevanza, che qui non è il criterio ma emerge comunque: articoli come quello sulla fine del sostegno USA all'offensiva in Yemen (2021-02-04) o sulla conferma del giudice Quraishi (2021-06-12) nominano ICE solo di passaggio.

In [ ]:
# 10 dei 21 candidati scartati dalla regola
show_sample(df[~df["is_ice_rule"]], 10)

**Come leggere la tabella.** Stesse colonne della precedente, per 10 dei 21 articoli scartati dalla regola. Una `menzioni` pari a 0 indica un risultato della query per frase senza alcuna menzione dell'agenzia nel testo. Un valore maggiore di 0 indica un articolo che nomina l'agenzia solo con la sigla, oppure con il nome completo in una forma che la regex non riconosce: è il caso dell'articolo del 2016-05-27 (22 menzioni, refuso "Immigration and Customers Enforcement").

### 4.1 Esito della validazione e limiti

**Disambiguazione.** 20 articoli su 20 si riferiscono all'agenzia federale: nessun falso positivo. Con un campione di 20 l'assenza di errori esclude un tasso di falsi positivi frequente, non uno raro: il limite superiore dell'intervallo di confidenza unilaterale al 95% (Clopper-Pearson) è 13,9%. La garanzia principale viene dalla costruzione della regola: il nome completo dell'agenzia e il tag editoriale non hanno altri referenti. Il campione conferma che non ci sono errori sistematici di implementazione.

**Rilevanza.** Almeno 4 dei 20 articoli trattano altro e citano ICE di passaggio: il Yemen e la conferma del giudice Quraishi nella tabella sopra, poi la nomina di Tom Homan commentata in un pezzo su Fox News (2024-11-14) e la riforma dei collegi elettorali in California (2025-08-19). È la motivazione diretta del filtro di rilevanza del §5.

**Articoli scartati.** Un controllo successivo (2026-10-08) ha esaminato tutti i 21 scartati, non solo i 10 della tabella:

- 17 non contengono il nome completo. Molti non nominano l'agenzia, altri sono articoli su ICE che usano solo la sigla (es. il raid nello stabilimento Hyundai in Georgia, 2025-09-12). La regola li perde per costruzione, perché la sigla non è un criterio ammesso;
- 1 contiene il refuso "Immigration and Customers Enforcement" (2016-05-27);
- 3 contengono il nome completo spezzato da caratteri Unicode invisibili (2026, vedi §2), e quindi sono falsi negativi da correggere.

**Limite sul recall.** Il controllo sugli scartati misura solo cosa la regola perde tra i candidati. Gli articoli che nessuna delle due query restituisce (senza tag e con la sola sigla nel testo) non entrano mai nella pipeline, e la loro quantità non si può stimare da qui. Sul NYT la stima era possibile perché si disponeva dell'intero archivio (notebook 01: circa il 5% di articoli ICE mancati nel 2025-2026); sul Guardian scaricare l'intero archivio non è praticabile con il limite di 500 richieste al giorno. La direzione dell'errore è comunque nota: mancano articoli che usano solo la sigla e non hanno il tag, quindi soprattutto articoli anteriori al 2025, quando il tag non veniva applicato. Il criterio resta coerente con la scelta di precisione rispetto al recall adottata per il NYT.

<!-- s5-relevance -->
## 5. Filtro di rilevanza: articolo su ICE o citazione di passaggio

La regola del §3 stabilisce che l'agenzia è nominata, non che l'articolo ne tratta. Il 33,9% degli articoli accettati la nomina al più due volte, e il campione del §4 contiene articoli sullo Yemen, su una nomina giudiziaria e sui collegi elettorali in cui ICE è un dettaglio. Usare il corpus della regola per le analisi tematiche mescolerebbe la copertura di ICE con quella di altri argomenti.

Una soglia fissa sul numero di menzioni sarebbe arbitraria: un articolo lungo su ICE può nominare l'agenzia poche volte, un live-blog molto lungo può nominarla spesso di sfuggita. Il filtro è quindi un classificatore supervisionato: gold set annotato a mano → modello → probabilità per tutti gli articoli del corpus.

**Definizione operativa (codebook)**, da citare nel report:

| Etichetta | Criterio |
|---|---|
| `y`, su ICE | L'agenzia, le sue azioni (arresti, raid, detenzioni, espulsioni), il suo personale, le sue strutture o le sue politiche sono il tema o uno dei temi centrali dell'articolo. Sono inclusi gli articoli su un singolo caso di detenzione o espulsione in cui ICE è l'attore. |
| `n`, di passaggio | ICE compare come contesto, in un elenco, come esempio o in una citazione, ma l'articolo tratta altro (politica estera, nomine, elezioni, cultura, sport). |
| `p`, parziale | Caso di confine: ICE ha un ruolo secondario ma non trascurabile, ad esempio un articolo sulla politica migratoria in generale con un paragrafo sull'agenzia. |

L'etichetta `p` rende esplicita l'incertezza invece di forzare una scelta binaria sui casi di confine. Gli articoli `p` sono esclusi dall'addestramento (`PARTIAL_AS = None`); il §5.6 mostra cosa cambierebbe unendoli a `y` o a `n`.

**Niente circolarità.** Etichette e feature vengono solo da articoli del Guardian; nessuna keyword del NYT entra nel filtro. Il confronto tra le due testate non dipende così da una definizione di rilevanza presa da una delle due.

**Annotazioni.** Le etichette manuali sono un input umano, non un output della pipeline: stanno in `data/annotations/`, separate da `data/processed/`, e il file non viene mai ricampionato né sovrascritto.

Struttura: §5.1 feature → §5.2 campione stratificato → §5.3 annotazione → §5.4 modelli e valutazione → §5.5 applicazione al corpus → §5.6 analisi di sensibilità → §5.7 verifica con validazione incrociata annidata.

<!-- s5-relevance -->
### 5.1 Feature

Il modello A usa 14 feature calcolate sul testo e sui metadati di ogni articolo accettato dalla regola. Il `body_text` del Guardian è una stringa senza a capo: i paragrafi non si possono ricostruire, e la posizione delle menzioni è espressa come posizione relativa nel testo (0 = inizio, 1 = fine).

| Feature | Definizione | Ipotesi |
|---|---|---|
| `log_mentions` | log(1 + menzioni) | più spazio all'agenzia, più rilevanza |
| `density` | menzioni ogni 1.000 parole | come sopra, al netto della lunghezza |
| `in_headline`, `in_standfirst` | menzione nel titolo, nel sottotitolo | il tema principale sta in apertura: probabilmente il segnale più forte |
| `first_pos` | posizione relativa della prima menzione | una menzione solo in fondo è tipica del passaggio |
| `in_lead` | prima menzione entro le prime 100 parole | come sopra |
| `spread` | (ultima − prima menzione) / lunghezza | un tema ricorrente è distribuito nel testo; una menzione isolata no |
| `has_ice_tag` | presenza del tag ICE | etichetta editoriale, sistematica solo dal 2025 |
| `is_live` | articolo di tipo live-blog | testi lunghissimi con molti argomenti |
| `log_words` | log(1 + parole) | i testi lunghi diluiscono il tema |
| `sec_*` | sezione: `us-news`, `commentisfree`, `world`, altro | profili di menzione diversi per sezione |

Le menzioni contano insieme nome completo e sigla. Dentro articoli già selezionati dalla regola del §3 la sigla si riferisce all'agenzia, e il vincolo di non selezionare mai sulla sigla resta rispettato, perché la selezione è già avvenuta. `has_ice_tag` è molto informativa ma esiste quasi solo dal 2025: un modello che vi si affidasse troppo funzionerebbe bene solo su quel periodo (verificato nel §5.6).

In [ ]:
import numpy as np
from IPython.core.error import StdinNotImplementedError
from IPython.display import clear_output, display

# Annotazioni manuali fuori da data/processed: sono input umano, non rigenerabile
ANNOTATIONS_DIR = BASE_DIR / "data" / "annotations"
GOLD_PATH = ANNOTATIONS_DIR / "guardian_relevance_gold.csv"
OUT_ICE_RELEVANT = PROCESSED_DIR / "articles_ice_relevant.parquet"

# Lo standfirst contiene HTML: i tag vanno tolti prima di cercare le menzioni
TAG_RE = re.compile(r"<[^>]+>")
# Attacco dell'articolo = prime 100 parole (feature in_lead)
LEAD_WORDS = 100


def agency_spans(text: str) -> list[int]:
    """Posizioni (offset in caratteri) di tutte le menzioni dell'agenzia nel testo, in ordine.

    Unisce le occorrenze del nome completo e della sigla. Usate per le feature di posizione
    e per le finestre di contesto dell'annotazione.
    """
    return sorted(m.start() for rx in (FULLNAME_RE, ACRONYM_RE) for m in rx.finditer(text))


def relevance_features(frame: pd.DataFrame) -> pd.DataFrame:
    """Calcola le 14 feature del modello A, una riga per articolo.

    Numero e densità delle menzioni, presenza nel titolo e nel sottotitolo, posizione relativa
    della prima menzione e distanza tra prima e ultima (rispetto alla lunghezza del testo),
    tag ICE, live-blog, lunghezza e sezione come variabili indicatrici. Definizioni nel §5.1.
    """
    body = frame["body_text"].fillna("")
    headline = frame["headline"].fillna("")
    standfirst = frame["standfirst"].fillna("").str.replace(TAG_RE, " ", regex=True)
    # Lunghezza dall'API; se manca, conteggio delle parole del testo
    words = frame["word_count"].fillna(body.str.split().str.len()).clip(lower=1)

    # Le posizioni diventano relative (0 = inizio, 1 = fine) dividendo per la lunghezza: il body_text non ha paragrafi
    spans = body.map(agency_spans)
    length = body.str.len().clip(lower=1)
    first = spans.map(lambda s: s[0] if s else np.nan)
    last = spans.map(lambda s: s[-1] if s else np.nan)
    lead_chars = body.map(lambda b: len(" ".join(b.split()[:LEAD_WORDS])))
    in_text = lambda s: s.str.contains(FULLNAME_RE) | s.str.contains(ACRONYM_RE)

    # Sezioni minori raggruppate in 'other', per non creare feature con pochissimi articoli
    section = frame["section_id"].where(frame["section_id"].isin(["us-news", "commentisfree", "world"]), "other")
    feats = pd.DataFrame({
        "log_mentions": np.log1p(frame["agency_mentions"]),
        "density": 1000 * frame["agency_mentions"] / words,
        "in_headline": in_text(headline).astype(int),
        "in_standfirst": in_text(standfirst).astype(int),
        # Senza menzioni nel testo (solo tag) la prima menzione è trattata come se fosse in fondo
        "first_pos": (first / length).fillna(1.0),
        "in_lead": (first < lead_chars).astype(int),
        "spread": ((last - first) / length).fillna(0.0),
        "has_ice_tag": frame["has_ice_tag"].astype(int),
        "is_live": frame["id"].str.contains("/live/", regex=False).astype(int),
        "log_words": np.log1p(words),
    }, index=frame.index)
    return feats.join(pd.get_dummies(section, prefix="sec", dtype=int))


# Si riparte dal parquet della regola: il §5 si può rieseguire senza il parsing
ice = pd.read_parquet(OUT_ICE_RULE)
ice["period"] = pd.cut(ice["year"], [2002, 2016, 2020, 2024, 9999], labels=[p[0] for p in PERIODS])
X_all = relevance_features(ice)
log.info("Feature di rilevanza calcolate: %d articoli x %d feature", *X_all.shape)
X_all.describe().T.round(2)

**Come leggere la tabella.** Statistiche descrittive delle 14 feature sui 3.292 articoli accettati dalla regola. Alcuni valori descrivono il corpus: il tag ICE è presente nel 36% degli articoli (quasi tutti del 2025-2026), il 78% è nella sezione `us-news`, il 12% sono live-blog, ICE è nel titolo nel 26% dei casi e la prima menzione cade in mediana al 21% del testo.

<!-- s5-relevance -->
### 5.2 Campione stratificato da annotare

Un campione casuale semplice servirebbe poco, per due motivi. Sarebbe composto per il 69% da articoli del 2025-2026, con pochi esempi per i periodi precedenti. E sarebbe dominato da articoli chiaramente rilevanti, mentre un classificatore impara soprattutto dai casi vicini al confine. Il campione è quindi stratificato su periodo × fascia di menzioni (1 / 2 / 3-5 / 6+): 16 celle, `PER_CELL = 13` articoli per cella, 208 in totale. Gli articoli con il solo tag e nessuna menzione nel testo finiscono nella fascia più bassa. L'ordine di annotazione è mescolato, così la stanchezza dell'annotatore non si correla con periodo o fascia.

Il campione si crea una sola volta (`SAMPLE_SEED = 42`). Se `GOLD_PATH` esiste viene riletto e mai ricampionato, perché le etichette già date resterebbero senza articolo. Per ampliarlo si usa `extend_gold()`, che aggiunge solo articoli nuovi.

**Pesi.** Ogni articolo annotato ha peso `w` = articoli della sua cella nel corpus / articoli annotati della stessa cella. I pesi riportano le metriche calcolate sul campione stratificato alla distribuzione reale del corpus (§5.4).

In [ ]:
# 13 articoli per cella × 16 celle (4 periodi × 4 fasce di menzioni) = 208 da annotare
PER_CELL = 13
SAMPLE_SEED = 42
# Fasce di menzioni 1 / 2 / 3-5 / 6+ (intervalli chiusi a destra)
MENTION_BINS = [0, 1, 2, 5, np.inf]
MENTION_LABELS = ["1", "2", "3-5", "6+"]

ice["mention_bin"] = pd.cut(ice["agency_mentions"], MENTION_BINS, labels=MENTION_LABELS)
# 0 menzioni = articolo con solo il tag (testo senza nome né sigla): finisce nella fascia più bassa
ice["mention_bin"] = ice["mention_bin"].fillna("1")
# Articoli del corpus in ogni cella: servono per i pesi delle metriche (§5.4)
cell_sizes = ice.groupby(["period", "mention_bin"], observed=False).size()


def draw_gold_sample(exclude_ids: set[str], per_cell: int, seed: int) -> pd.DataFrame:
    """Estrae il campione da annotare, stratificato per periodo × fascia di menzioni.

    Esclude gli id in exclude_ids, prende fino a per_cell articoli per cella con il seed dato e
    mescola l'ordine finale, così la stanchezza dell'annotatore non si lega a periodo o fascia.
    Restituisce la tabella da annotare, con le colonne label e note vuote.
    """
    pool = ice[~ice["id"].isin(exclude_ids)]
    parts = [g.sample(min(per_cell, len(g)), random_state=seed) for _, g in pool.groupby(["period", "mention_bin"], observed=True)]
    s = pd.concat(parts).sample(frac=1, random_state=seed)
    return pd.DataFrame({
        "id": s["id"], "period": s["period"].astype(str), "mention_bin": s["mention_bin"].astype(str),
        "headline": s["headline"], "web_url": s["web_url"], "label": "", "note": "",
    })


def load_or_create_gold() -> pd.DataFrame:
    """Restituisce il gold set: lo rilegge se il file esiste, altrimenti lo estrae e lo salva.

    Un gold set esistente non viene mai ricampionato, perché le etichette già date
    resterebbero senza il loro articolo.
    """
    if GOLD_PATH.exists():
        gold = pd.read_csv(GOLD_PATH, dtype=str, keep_default_na=False)
        log.info("Gold set esistente riletto: %d articoli, %d annotati", len(gold), (gold["label"] != "").sum())
        return gold
    gold = draw_gold_sample(set(), PER_CELL, SAMPLE_SEED)
    ANNOTATIONS_DIR.mkdir(parents=True, exist_ok=True)
    gold.to_csv(GOLD_PATH, index=False)
    log.info("Nuovo gold set campionato: %d articoli -> %s", len(gold), GOLD_PATH.relative_to(BASE_DIR))
    return gold


def extend_gold(per_cell: int, seed: int = SAMPLE_SEED + 1) -> pd.DataFrame:
    """Amplia il gold set con per_cell articoli nuovi per cella, senza toccare quelli già presenti.

    Usa un seed diverso da quello originale ed esclude gli id già nel file; salva il CSV aggiornato.
    """
    gold = load_or_create_gold()
    extra = draw_gold_sample(set(gold["id"]), per_cell, seed)
    gold = pd.concat([gold, extra], ignore_index=True)
    gold.to_csv(GOLD_PATH, index=False)
    log.info("Gold set esteso di %d articoli (totale %d)", len(extra), len(gold))
    return gold


gold = load_or_create_gold()
pd.crosstab(gold["period"], gold["mention_bin"], margins=True).reindex(columns=MENTION_LABELS + ["All"])

**Come leggere la tabella.** Articoli del gold set per periodo (righe) e fascia di menzioni (colonne): 13 per cella, come da disegno, 208 in totale. La tabella verifica che il campionamento abbia riempito tutte le celle, cioè che ogni cella del corpus contenga almeno 13 articoli.

<!-- s5-relevance -->
### 5.3 Annotazione

La cella seguente è uno strumento di annotazione interattivo. Per ogni articolo mostra data, sezione, lunghezza, numero di menzioni, presenza del tag, titolo, sottotitolo, le prime 70 parole e fino a 4 finestre di contesto attorno alle menzioni, distribuite lungo il testo e con la posizione relativa di ciascuna: si vede subito se ICE ritorna nel pezzo o compare una volta sola. Tasti: `y` su ICE · `n` di passaggio · `p` parziale · `s` salta · `q` esci. Dopo l'etichetta si può aggiungere una nota libera, separata da uno spazio (`n yemen`).

Il file viene salvato dopo ogni etichetta, quindi l'annotazione si interrompe e riprende dal primo articolo non annotato. In esecuzione non interattiva (nbconvert, *Run All* senza input) la cella stampa solo lo stato.

**Come è stato annotato il gold set.** Il 2026-09-30, direttamente sul CSV (colonna `label` ∈ {`y`, `n`, `p`}), aprendo ogni articolo completo dal `web_url`; la cella interattiva non è stata usata. Un solo annotatore, senza misura di accordo tra annotatori.

In [ ]:
VALID_LABELS = {"y", "n", "p"}
ice_by_id = ice.set_index("id")


def mention_windows(body: str, k: int = 4, width: int = 180) -> list[str]:
    """Restituisce fino a k finestre di testo attorno alle menzioni, distribuite lungo l'articolo.

    Sceglie menzioni equidistanti nell'elenco (la prima, l'ultima e alcune intermedie) e antepone
    a ogni finestra la sua posizione relativa nel testo: l'annotatore vede subito se ICE ritorna
    nel pezzo o compare una volta sola.
    """
    spans = agency_spans(body)
    if not spans:
        return []
    picks = [spans[i] for i in sorted({round(j * (len(spans) - 1) / max(k - 1, 1)) for j in range(k)})]
    return [f"[{100 * p / max(len(body), 1):.0f}%] …{body[max(0, p - width): p + width]}…" for p in picks]


def show_article(row: pd.Series, done: int, total: int) -> None:
    """Stampa la scheda di un articolo per l'annotazione.

    Metadati (data, sezione, lunghezza, menzioni, tag), titolo, sottotitolo, prime 70 parole,
    finestre di contesto di mention_windows e URL dell'articolo.
    """
    a = ice_by_id.loc[row["id"]]
    standfirst = TAG_RE.sub(" ", a["standfirst"] or "").strip()
    print(f"[{done}/{total}]  {a['pub_date'].date()}  ·  {a['section_id']}  ·  {a['word_count']} parole  ·  "
          f"menzioni {a['agency_mentions']}  ·  tag ICE {'sì' if a['has_ice_tag'] else 'no'}")
    print(f"\n### {a['headline']}\n{standfirst}\n")
    print("ATTACCO:", " ".join(a["body_text"].split()[:70]), "…\n")
    print(f"MENZIONI ({len(agency_spans(a['body_text']))} totali, [posizione nel testo]):")
    for w in mention_windows(a["body_text"]):
        print("  •", w)
    print(f"\n{a['web_url']}")


def annotate(gold: pd.DataFrame) -> pd.DataFrame:
    """Ciclo di annotazione interattivo sugli articoli del gold set ancora senza etichetta.

    Per ogni articolo mostra la scheda e chiede y / n / p (con nota facoltativa), s per saltare
    o q per uscire. Il CSV viene salvato dopo ogni etichetta, quindi si può interrompere e riprendere.
    """
    todo = gold.index[gold["label"] == ""].tolist()
    total = len(gold)
    for i in todo:
        clear_output(wait=True)
        show_article(gold.loc[i], total - len(todo) + todo.index(i) + 1, total)
        while True:
            answer = input("\n[y] su ICE · [n] passaggio · [p] parziale · [s] salta · [q] esci > ").strip()
            key, _, note = answer.partition(" ")
            key = key.lower()
            if key in VALID_LABELS | {"s", "q"}:
                break
            print("Tasto non valido.")
        if key == "q":
            break
        if key == "s":
            continue
        gold.loc[i, ["label", "note"]] = [key, note]
        gold.to_csv(GOLD_PATH, index=False)
    clear_output(wait=True)
    return gold


# In esecuzione non interattiva input() non è disponibile: si stampa solo lo stato
try:
    gold = annotate(gold)
except (EOFError, StdinNotImplementedError):
    log.info("Esecuzione non interattiva: annotazione saltata")

print(f"Annotati {(gold['label'] != '').sum()}/{len(gold)}")
print(gold["label"].replace("", "(da fare)").value_counts().to_string())

**Come leggere l'output.** Lo stato conferma che tutti i 208 articoli sono annotati: 104 `n` (50%), 71 `y` (34%), 33 `p` (16%), quindi il training usa 175 articoli. La relazione tra numero di menzioni ed etichetta è forte ma non netta:

| Fascia di menzioni | `n` | `y` | `p` |
|---|---|---|---|
| 1 | 37 | 5 | 10 |
| 2 | 35 | 9 | 8 |
| 3-5 | 23 | 17 | 12 |
| 6+ | 9 | 40 | 3 |

Con una sola menzione gli articoli sono quasi tutti di passaggio, con sei o più quasi tutti rilevanti, ma nelle fasce intermedie le classi si sovrappongono, ed è lì che una soglia fissa sulle menzioni sbaglierebbe di più. Il tag ICE è quasi un sinonimo di rilevanza (37 dei 38 articoli `y`/`n` con il tag sono `y`), ma esiste quasi solo dal 2025.

<!-- s5-relevance -->
### 5.4 Modelli e valutazione

Due modelli, entrambi di regressione logistica: con 175 esempi un modello più flessibile sovradatterebbe, e la regressione logistica produce probabilità e coefficienti leggibili.

- **A, feature esplicite.** Regressione logistica sulle 14 feature costruite a mano del §5.1: le feature sono definite da noi, i pesi il modello li impara dai 175 articoli annotati `y`/`n`. È interpretabile: a ogni feature corrisponde un coefficiente, e la logica della decisione si descrive a parole ("ICE nel titolo alza la probabilità, una prima menzione in fondo al testo la abbassa").
- **B, feature + embedding.** Le stesse 14 feature più l'embedding a 384 dimensioni di `all-MiniLM-L6-v2` su titolo, sottotitolo e prime 250 parole (il modello tronca comunque l'input a 256 token). L'embedding riconosce un articolo su ICE dal contenuto anche quando le menzioni esplicite sono poche. Il prezzo è l'interpretabilità: le singole dimensioni non hanno un significato nominabile (la dimensione 217 non vuol dire "parla di detenzioni"), e di un articolo si può dire solo che il suo testo somiglia a quelli etichettati `y`, non quale caratteristica lo ha fatto tenere o scartare.

In entrambi i modelli le feature sono standardizzate, la regolarizzazione è L2 con `C = 0.5` e `class_weight="balanced"` compensa la prevalenza di `n` (104 contro 71). Gli embedding sono in cache (`relevance_embeddings_minilm.parquet`) e si ricalcolano solo se cambia l'insieme degli articoli.

**Valutazione.** Validazione incrociata stratificata a 5 fold (`CV_SEED = 42`): ogni articolo riceve una probabilità da un modello che non lo ha visto in addestramento (previsioni *out-of-fold*). Su queste si calcolano ROC-AUC, average precision e, fissata la soglia, precisione e recall, tutte pesate con `w`. Precisione e recall sono calcolate anche per periodo, perché il filtro deve funzionare anche nel 2003-2016 e non solo dove gli esempi sono più numerosi.

**Soglia.** La più bassa che garantisce una precisione pesata ≥ `TARGET_PRECISION` (0,90) sulle previsioni out-of-fold: la stessa preferenza per la precisione del filtro NYT. Scegliere la soglia sulle stesse previsioni su cui si misurano le metriche può renderle ottimistiche; il §5.7 misura questa distorsione.

**Scelta del modello.** A è il default, perché è più facile da giustificare. Si adotta B se vale almeno uno dei due criteri:

1. qualità globale: l'average precision pesata di B supera quella di A di almeno `MIN_AP_GAIN_FOR_B` (0,03);
2. uniformità nel tempo: il recall minimo tra i periodi di B supera quello di A di almeno `MIN_RECALL_GAIN_FOR_B` (0,10).

Il criterio 2 è stato aggiunto dopo aver visto i risultati del primo run: è una scelta *post hoc* da dichiarare nel report. Con il solo criterio 1 veniva scelto A, che recupera il 57-69% degli articoli rilevanti prima del 2025 e il 95% dal 2025. Un filtro con recall diverso per periodo altera proprio i confronti diacronici per cui il corpus viene costruito, perché la crescita del 2025-2026 risulterebbe amplificata dal filtro stesso. L'uniformità del recall conta quindi più di un'average precision quasi identica. Il valore 0,10 non è ottimizzato; il guadagno osservato (+0,26) lo supera di oltre due volte e mezzo. Il confronto con le alternative scartate è nel §5.6.

**Il grafico dei coefficienti descrive A, non il filtro applicato.** È calcolato sempre sul modello A, l'unico con coefficienti leggibili. Quando la regola sceglie B, il grafico non descrive il filtro che ha prodotto `articles_ice_relevant.parquet`: va letto, e citato nel report, come indicazione qualitativa dei segnali che distinguono un articolo su ICE da una citazione di passaggio. La giustificazione di B passa dalle metriche in validazione incrociata, non dai coefficienti.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, precision_recall_curve, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Sotto questo numero di etichette utilizzabili il modello non viene addestrato
MIN_LABELS = 80
PARTIAL_AS = None      # None = 'p' escluso dal training; 0 o 1 per unirlo a 'n' o 'y'
# Vincolo per la soglia di decisione: precisione pesata ≥ 0,90 (precisione prima del recall, come per il NYT)
TARGET_PRECISION = 0.90
# Margini minimi della regola di scelta tra A e B (§5.4): non sono soglie di decisione
MIN_AP_GAIN_FOR_B = 0.03
# Il subset serve a confronti tra periodi: un recall disomogeneo nel tempo distorce il confronto più di un'AP leggermente inferiore
MIN_RECALL_GAIN_FOR_B = 0.10
# Stesso modello di embedding usato nei notebook 04 e 07
EMBED_MODEL = "all-MiniLM-L6-v2"
EMBED_CACHE = PROCESSED_DIR / "relevance_embeddings_minilm.parquet"
CV_SEED = 42
for noisy in ("httpx", "sentence_transformers", "huggingface_hub"):
    logging.getLogger(noisy).setLevel(logging.WARNING)

# Feature indicizzate per id, per allinearle a gold set ed embedding
X_all.index = ice["id"].values
labeled = gold[gold["label"].isin(VALID_LABELS)].copy()
# Dimensione delle celle nel corpus, con chiave (periodo, fascia)
corpus_cells = {(str(p), str(b)): n for (p, b), n in cell_sizes.items()}


def build_train(partial_as) -> pd.DataFrame:
    """Costruisce il training set dalle etichette manuali, con target y e peso w.

    partial_as decide cosa fare delle p: None le esclude, 1 le conta come y, 0 come n.
    Il peso w = articoli della cella nel corpus / articoli annotati della cella riporta
    le metriche del campione stratificato alla distribuzione reale del corpus.
    """
    data = labeled.assign(y=labeled["label"].map({"y": 1, "n": 0, "p": partial_as}))
    data = data.dropna(subset=["y"]).astype({"y": int}).reset_index(drop=True)
    # Peso = articoli della cella nel corpus / articoli annotati della cella: riporta il campione stratificato al corpus reale
    labeled_cells = data.groupby(["period", "mention_bin"]).size()
    data["w"] = [corpus_cells[(p, b)] / labeled_cells[(p, b)] for p, b in zip(data["period"], data["mention_bin"])]
    return data


train = build_train(PARTIAL_AS)
# Si procede solo con abbastanza etichette e con entrambe le classi presenti
ready = len(train) >= MIN_LABELS and train["y"].nunique() == 2


def embed_corpus() -> pd.DataFrame:
    """Restituisce gli embedding MiniLM (384 dimensioni) degli articoli della regola, una riga per id.

    Testo = titolo + sottotitolo + prime 250 parole. Usa la cache su disco se contiene esattamente
    gli stessi articoli; altrimenti ricalcola gli embedding e aggiorna la cache.
    """
    # Cache su disco: ~3k testi su CPU richiedono qualche minuto; si ricalcola solo se cambia l'insieme di articoli
    if EMBED_CACHE.exists():
        cached = pd.read_parquet(EMBED_CACHE)
        if set(cached.index) == set(ice["id"]):
            return cached.loc[ice["id"]]
    from sentence_transformers import SentenceTransformer
    # Testo per l'embedding: titolo + sottotitolo + prime 250 parole (MiniLM tronca comunque a 256 token)
    texts = (
        ice["headline"].fillna("") + ". "
        + ice["standfirst"].fillna("").str.replace(TAG_RE, " ", regex=True) + " "
        + ice["body_text"].fillna("").map(lambda b: " ".join(b.split()[:250]))
    ).tolist()
    log.info("Embedding di %d articoli con %s...", len(texts), EMBED_MODEL)
    vecs = SentenceTransformer(EMBED_MODEL).encode(texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
    emb = pd.DataFrame(vecs, index=ice["id"].values, columns=[f"e{i}" for i in range(vecs.shape[1])])
    emb.to_parquet(EMBED_CACHE)
    return emb


def make_model():
    """Crea un modello non addestrato: standardizzazione delle feature + regressione logistica.

    Regolarizzazione L2 con C = 0.5, moderata e adatta a pochi esempi; class_weight="balanced"
    compensa lo squilibrio tra n (104) e y (71). È lo stesso modello per A e B: cambiano solo le feature.
    """
    return make_pipeline(StandardScaler(), LogisticRegression(C=0.5, class_weight="balanced", max_iter=5000))


def pick_threshold(y, proba, w) -> float:
    """Sceglie la soglia di decisione: la più bassa con precisione pesata ≥ TARGET_PRECISION.

    Scorre la curva precisione-recall delle probabilità, pesata con w: tra le soglie che rispettano
    il vincolo, la più bassa è quella con il recall più alto. Se nessuna lo rispetta, usa quella
    a precisione massima e lo segnala nel log.
    """
    prec, rec, thr = precision_recall_curve(y, proba, sample_weight=w)
    ok = np.where(prec[:-1] >= TARGET_PRECISION)[0]
    if len(ok) == 0:
        log.warning("Nessuna soglia raggiunge precisione %.2f: uso quella a precisione massima", TARGET_PRECISION)
        return float(thr[np.argmax(prec[:-1])])
    return float(thr[ok[0]])


def evaluate(name: str, X: pd.DataFrame, data: pd.DataFrame) -> tuple[dict, pd.DataFrame]:
    """Valuta un modello in validazione incrociata stratificata a 5 fold.

    Calcola le probabilità out-of-fold, sceglie la soglia con pick_threshold e misura precisione
    e recall pesati, sul corpus e per periodo. Restituisce (riepilogo con soglia, ROC-AUC e
    average precision; tabella delle metriche per gruppo).
    """
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=CV_SEED)
    # Previsioni out-of-fold: ogni articolo è valutato da un modello che non lo ha visto
    proba = cross_val_predict(make_model(), X.values, data["y"].values, cv=cv, method="predict_proba")[:, 1]
    y, w = data["y"].values, data["w"].values
    thr = pick_threshold(y, proba, w)
    pred = (proba >= thr).astype(int)
    rows = []
    # Metriche pesate sul corpus intero e su ciascun periodo
    for period, idx in [("corpus (pesato)", data.index)] + list(data.groupby("period").groups.items()):
        yy, pp, ww = y[idx], pred[idx], w[idx]
        rows.append({
            "modello": name, "gruppo": period, "n": len(idx), "positivi": int(yy.sum()),
            "precision": precision_score(yy, pp, sample_weight=ww, zero_division=0),
            "recall": recall_score(yy, pp, sample_weight=ww, zero_division=0),
        })
    summary = {
        "modello": name, "soglia": thr,
        "ROC-AUC": roc_auc_score(y, proba, sample_weight=w),
        "avg precision": average_precision_score(y, proba, sample_weight=w),
    }
    return summary, pd.DataFrame(rows)


if not ready:
    log.warning("Annotazioni insufficienti per il modello: %d utilizzabili (servono >= %d, entrambe le classi). "
                "Completa §5.3 e riesegui da qui.", len(train), MIN_LABELS)
else:
    log.info("Training su %d articoli (%d su ICE, %d di passaggio; %d parziali esclusi)",
             len(train), train["y"].sum(), (train["y"] == 0).sum(), (labeled["label"] == "p").sum() if PARTIAL_AS is None else 0)

    emb_all = embed_corpus()
    XA = X_all.loc[train["id"]]
    XB = XA.join(emb_all)

    MODEL_A, MODEL_B = "A: feature", "B: feature + embedding"
    results = [evaluate(MODEL_A, XA, train), evaluate(MODEL_B, XB, train)]
    summaries = pd.DataFrame([r[0] for r in results]).set_index("modello")
    per_period = pd.concat([r[1] for r in results])

    ap_gain = summaries.loc[MODEL_B, "avg precision"] - summaries.loc[MODEL_A, "avg precision"]
    min_recall = per_period[per_period["gruppo"] != "corpus (pesato)"].groupby("modello")["recall"].min()
    recall_gain = min_recall[MODEL_B] - min_recall[MODEL_A]
    # Regola di scelta: B se supera A di almeno uno dei due margini minimi, altrimenti A
    CHOSEN = MODEL_B if ap_gain >= MIN_AP_GAIN_FOR_B or recall_gain >= MIN_RECALL_GAIN_FOR_B else MODEL_A
    THRESHOLD = summaries.loc[CHOSEN, "soglia"]
    log.info("Guadagno di B su A — AP: %+.3f (soglia %.2f) · recall minimo tra periodi: %.3f -> %.3f = %+.3f (soglia %.2f)",
             ap_gain, MIN_AP_GAIN_FOR_B, min_recall[MODEL_A], min_recall[MODEL_B], recall_gain, MIN_RECALL_GAIN_FOR_B)
    log.info("Modello scelto: %s, soglia %.3f", CHOSEN, THRESHOLD)
    display(summaries.round(3))
    display(per_period.round(3).pivot(index="gruppo", columns="modello", values=["precision", "recall"]))

    # Modello A addestrato su tutto il gold set solo per leggerne i coefficienti
    coef_model = make_model().fit(XA.values, train["y"].values)
    coefs = pd.Series(coef_model[-1].coef_[0], index=XA.columns).sort_values()
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.barh(coefs.index, coefs.values, color=np.where(coefs.values > 0, "#2a78d6", "#eb6834"), height=0.6)
    ax.axvline(0, color="#999999", linewidth=0.8)
    # I coefficienti sono leggibili solo per A: se il filtro applicato è B il grafico resta come lettura qualitativa, e va detto
    role = "the model applied to the corpus" if CHOSEN == MODEL_A else "qualitative reading: the model applied to the corpus is B"
    ax.set_title(f"Model A: standardized coefficients ({role})\n(> 0 pushes towards 'about ICE', < 0 towards 'passing mention')", fontsize=10)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    fig.savefig(FIG_DIR / "guardian_model_a_coefficients.png", dpi=200, bbox_inches="tight")
    plt.show()

**Come leggere l'output.**

- **Log.** I due guadagni di B su A: +0,013 di average precision (sotto la soglia di 0,03) e +0,259 di recall minimo tra i periodi (da 0,572 a 0,831, sopra la soglia di 0,10). Modello scelto: B, soglia 0,245.
- **Prima tabella, metriche globali.** I modelli sono vicini (ROC-AUC 0,960 contro 0,978, average precision 0,973 contro 0,986). Le soglie non sono confrontabili tra loro: ciascuna è scelta sulla distribuzione delle probabilità del proprio modello.
- **Seconda tabella, precisione e recall per periodo.** Qui i modelli si separano: il recall di A è 0,57-0,69 prima del 2025 e 0,95 dopo, quello di B sta tra 0,83 e 1,00 in tutti i periodi. B paga con una precisione più irregolare (0,715 nel 2021-2024). Ogni stima per periodo poggia su 14-24 articoli `y`: gli intervalli di confidenza sono nel grafico successivo.
- **Grafico dei coefficienti** (modello A, feature standardizzate). Barre blu: spingono verso "su ICE"; arancioni: verso "di passaggio". Il segnale più forte è ICE nel titolo (+1,10), seguito da tag, distribuzione e numero delle menzioni; spingono verso il passaggio la lunghezza dell'articolo, le sezioni minori e il formato live-blog. Il coefficiente di `in_lead` è negativo (−0,25), contro l'intuizione: `in_lead` è fortemente correlata con `first_pos` e `in_headline`, e con feature correlate i singoli coefficienti non si interpretano separatamente.

In [ ]:
# Intervalli per periodo: bootstrap degli articoli annotati entro ciascun periodo, sulle stesse
# previsioni out-of-fold della tabella (stesso CV_SEED, stessa soglia di ciascun modello)
N_BOOT = 2000

if ready:
    rng = np.random.default_rng(CV_SEED)
    y_tr, w_tr = train["y"].values, train["w"].values
    ci_rows = []
    for name, X in [(MODEL_A, XA), (MODEL_B, XB)]:
        # Stesso seed e stessi fold del §5.4: previsioni identiche a quelle della tabella
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=CV_SEED)
        proba = cross_val_predict(make_model(), X.values, y_tr, cv=cv, method="predict_proba")[:, 1]
        pred = (proba >= summaries.loc[name, "soglia"]).astype(int)
        for period, _, _ in PERIODS:
            idx = np.where((train["period"] == period).values)[0]
            for metric_name, metric in [("precision", precision_score), ("recall", recall_score)]:
                point = metric(y_tr[idx], pred[idx], sample_weight=w_tr[idx], zero_division=0)
                boot = []
                for _ in range(N_BOOT):
                    # Ricampionamento con reinserimento degli articoli annotati del periodo
                    b = rng.choice(idx, len(idx))
                    # Senza articoli y il recall non è definito: il ricampionamento si scarta
                    if y_tr[b].sum() > 0:
                        boot.append(metric(y_tr[b], pred[b], sample_weight=w_tr[b], zero_division=0))
                lo, hi = np.percentile(boot, [2.5, 97.5])
                # A 1,00 il bootstrap non varia: limite inferiore unilaterale di Clopper-Pearson sui conteggi non pesati
                n_den = int(y_tr[idx].sum()) if metric_name == "recall" else int(pred[idx].sum())
                cp_lo = 0.025 ** (1 / n_den) if point == 1.0 and hi - lo < 1e-9 and n_den > 0 else np.nan
                ci_rows.append({"model": name[0], "period": period, "metric": metric_name, "value": point,
                                "lo": lo, "hi": hi, "cp_lo": cp_lo, "n": n_den})
    ci = pd.DataFrame(ci_rows)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
    colors = {"A": "#eb6834", "B": "#2a78d6"}
    x = np.arange(len(PERIODS))
    for ax, metric_name in zip(axes, ["precision", "recall"]):
        for offset, model in [(-0.12, "A"), (0.12, "B")]:
            part = ci[(ci["metric"] == metric_name) & (ci["model"] == model)]
            ax.errorbar(x + offset, part["value"], yerr=[part["value"] - part["lo"], part["hi"] - part["value"]],
                        fmt="o", color=colors[model], capsize=3, label=f"model {model}")
            for xi, row in zip(x + offset, part.itertuples()):
                if not np.isnan(row.cp_lo):
                    ax.vlines(xi, row.cp_lo, 1.0, colors=colors[model], linestyles=":", linewidth=1.5)
                    ax.hlines(row.cp_lo, xi - 0.05, xi + 0.05, colors=colors[model], linewidth=1.2)
        if metric_name == "precision":
            ax.axhline(TARGET_PRECISION, color="#999999", linewidth=0.8, linestyle="--")
            ax.text(x[-1] + 0.35, TARGET_PRECISION, "target\n(corpus)", fontsize=8, color="#666666", va="center")
        ax.set_title(metric_name.capitalize())
        ax.set_xticks(x, [p for p, _, _ in PERIODS])
        ax.set_ylim(0, 1.05)
        ax.spines[["top", "right"]].set_visible(False)
    axes[0].legend(frameon=False, loc="lower left")
    fig.suptitle("Relevance filter: out-of-fold precision and recall by period\n"
                 "(solid: 95% bootstrap interval; dotted: Clopper-Pearson lower bound where the estimate is 1.00)", fontsize=10)
    plt.tight_layout()
    fig.savefig(FIG_DIR / "guardian_relevance_by_period.png", dpi=200, bbox_inches="tight")
    plt.show()
    display(ci.pivot_table(index=["metric", "period"], columns="model", values=["value", "lo", "hi", "cp_lo", "n"]).round(3))

**Come leggere il grafico.** Precisione (sinistra) e recall (destra) per periodo, in validazione incrociata, per il modello A (arancione) e B (blu). I punti sono i valori della tabella del §5.4; le barre continue sono intervalli bootstrap al 95% (2.000 ricampionamenti degli articoli annotati entro il periodo). Quando la stima è 1,00 il bootstrap non varia, perché ogni ricampionamento dà ancora 1,00, e la barra continua sparirebbe dando un'impressione di certezza: in quei casi la linea tratteggiata arriva al limite inferiore unilaterale al 95% di Clopper-Pearson sui conteggi non pesati (es. recall di B: 0,82 nel 2003-2016 su 18 articoli `y`, 0,78 nel 2021-2024 su 15). La riga grigia è il vincolo di precisione 0,90, che vale sul corpus pesato e non in ciascun periodo. La tabella sotto il grafico riporta tutti i valori.

Gli intervalli sono larghi: ogni periodo ha 14-24 articoli `y`. Presa periodo per periodo, la differenza di recall tra A e B è indicativa più che conclusiva, perché gli intervalli si sfiorano nel 2003-2016 e nel 2021-2024 e si sovrappongono nel 2017-2020. Il dato solido è la coerenza dello schema: in tutti e tre i periodi anteriori al 2025 il recall di A è più basso di quello di B, mentre dal 2025 i due modelli coincidono. Il costo di B si vede a sinistra: la precisione nel 2021-2024 (0,715, intervallo 0,49-0,90) è la più bassa e la più incerta. La figura è salvata in `docs/figures/guardian_relevance_by_period.png`.

**Colonne della tabella.** Una riga per metrica e periodo; ogni colonna ha un valore per il modello A e uno per il B. Le colonne sono in ordine alfabetico, non di lettura.

| Colonna | Significato |
|---|---|
| `value` | stima puntuale (precisione o recall del periodo, out-of-fold, pesata entro il periodo): i pallini del grafico e i valori della tabella per periodo del §5.4 |
| `lo`, `hi` | estremi dell'intervallo bootstrap al 95% (2,5° e 97,5° percentile dei 2.000 ricampionamenti): le barre continue |
| `cp_lo` | limite inferiore unilaterale al 95% di Clopper-Pearson, `0,025^(1/n)`, calcolato solo quando la stima è 1,00 e il bootstrap non varia (`lo` = `hi` = 1); altrimenti `NaN`. È la linea tratteggiata |
| `n` | denominatore della metrica, non pesato: per il recall gli articoli `y` del periodo, per la precisione gli articoli che il modello ha classificato come rilevanti |

Esempio: recall 2003-2016 del modello B, `value` 1,000, `lo` = `hi` = 1,000, `n` 18, `cp_lo` 0,815. Il modello trova tutti i 18 articoli `y`, il bootstrap non può dire nulla sull'incertezza, e Clopper-Pearson indica che un recall reale fino a 0,82 è ancora compatibile con 18 successi su 18. Per il modello A nello stesso periodo: `value` 0,624, intervallo da 0,383 a 0,856 su 18 articoli.

<!-- s5-relevance -->
### 5.5 Applicazione al corpus

Il modello scelto viene riaddestrato su tutti i 175 articoli annotati `y`/`n` e applicato ai 3.292 articoli di `articles_ice_rule.parquet`, che resta intatto. L'output `articles_ice_relevant.parquet` contiene tutti gli articoli, con tre colonne in più:

- `relevance_score`: probabilità prevista, tra 0 e 1;
- `is_ice_relevant`: `relevance_score ≥ THRESHOLD`. La decisione è sempre del modello, anche per gli articoli annotati, così il criterio è uniforme su tutto il corpus;
- `gold_label`: l'etichetta manuale, dove esiste, per trasparenza e controlli successivi.

La cella mostra poi quattro output: gli articoli tenuti per periodo, un grafico per anno a due pannelli, il confronto tra decisione del modello ed etichette manuali e i 12 articoli più vicini alla soglia.

In [ ]:
if not ready:
    log.warning("§5.5 saltata: modello non addestrato (vedi §5.4)")
else:
    # Modello finale dello stesso tipo di quello scelto, riaddestrato su tutti i 175 articoli y/n
    X_final = X_all if CHOSEN.startswith("A") else X_all.join(emb_all)
    final_model = make_model().fit(X_final.loc[train["id"]].values, train["y"].values)

    # Si tengono tutti gli articoli della regola e si aggiungono score, decisione ed etichetta manuale
    out = ice.drop(columns=["mention_bin"]).copy()
    out["relevance_score"] = final_model.predict_proba(X_final.loc[out["id"]].values)[:, 1]
    out["is_ice_relevant"] = out["relevance_score"] >= THRESHOLD
    out["gold_label"] = out["id"].map(gold.set_index("id")["label"]).replace("", pd.NA)
    out["period"] = out["period"].astype(str)
    out.to_parquet(OUT_ICE_RELEVANT, index=False)
    log.info("Scritto %s: %d articoli, %d rilevanti (modello %s, soglia %.3f)",
             OUT_ICE_RELEVANT.relative_to(BASE_DIR), len(out), out["is_ice_relevant"].sum(), CHOSEN, THRESHOLD)

    kept = out.groupby("period").agg(articoli=("id", "size"), rilevanti=("is_ice_relevant", "sum"))
    kept["% tenuti"] = (100 * kept["rilevanti"] / kept["articoli"]).round(1)
    display(kept)

    yearly_rel = out.groupby("year").agg(regola=("id", "size"), rilevanti=("is_ice_relevant", "sum"))
    yearly_rel["pct"] = 100 * yearly_rel["rilevanti"] / yearly_rel["regola"]
    # Sotto questa numerosità la quota di un singolo anno è rumore (1 articolo sposta la % di oltre 10 punti)
    MIN_YEAR_N = 10
    reliable = (yearly_rel["regola"] >= MIN_YEAR_N).values
    overall_pct = 100 * out["is_ice_relevant"].mean()

    # Due pannelli con asse x condiviso invece di un secondo asse y: conteggi e quote hanno scale diverse
    fig, (ax, ax_pct) = plt.subplots(2, 1, figsize=(11, 6.5), sharex=True, gridspec_kw={"height_ratios": [3, 2]})
    yearly_rel[["regola", "rilevanti"]].plot(kind="bar", ax=ax, color=["#b7d3f6", "#2a78d6"], width=0.8)
    ax.set_title("Guardian ICE articles per year: rule filter vs relevance filter")
    ax.set_ylabel("articles")
    ax.legend(["rule filter (agency named)", f"relevant (score ≥ {THRESHOLD:.3f})"], frameon=False)
    ax.spines[["top", "right"]].set_visible(False)

    x = np.arange(len(yearly_rel))
    ax_pct.bar(x, yearly_rel["pct"], width=0.6, color=np.where(reliable, "#2a78d6", "#c9ccd1"))
    ax_pct.axhline(overall_pct, color="#666666", linewidth=0.8, linestyle="--")
    ax_pct.text(0.01, overall_pct + 2, f"whole corpus {overall_pct:.0f}%", transform=ax_pct.get_yaxis_transform(),
                fontsize=8, color="#555555", va="bottom")
    for xi, pct, ok in zip(x, yearly_rel["pct"], reliable):
        if ok:
            ax_pct.text(xi, pct + 2, f"{pct:.0f}%", ha="center", va="bottom", fontsize=8, color="#333333",
                            bbox={"facecolor": "white", "edgecolor": "none", "pad": 0.8})
    ax_pct.set_title(f"Share of articles kept by the relevance filter (grey: years with fewer than {MIN_YEAR_N} articles, unreliable share)",
                     fontsize=10)
    ax_pct.set_ylabel("% relevant")
    ax_pct.set_ylim(0, 100)
    ax_pct.set_xlabel("")
    ax_pct.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    fig.savefig(FIG_DIR / "guardian_relevance_by_year.png", dpi=200, bbox_inches="tight")
    plt.show()

    display(pd.crosstab(out["gold_label"].fillna("(non annotato)"), out["is_ice_relevant"], margins=True))

    # I 12 articoli con score più vicino alla soglia: i casi più incerti, da controllare a mano
    borderline = out.iloc[(out["relevance_score"] - THRESHOLD).abs().argsort()[:12]]
    display(borderline[["pub_date", "section_id", "agency_mentions", "relevance_score", "gold_label", "headline"]]
            .assign(pub_date=lambda d: d["pub_date"].dt.date).round(3))

**Come leggere gli output.**

- **Tabella per periodo.** 1.738 articoli rilevanti su 3.292 (52,8%): 49,3% nel 2003-2016, 49,8% nel 2017-2020, 39,3% nel 2021-2024, 55,2% dal 2025. Con il primo run (modello A) le quote erano 26,9 / 32,4 / 28,4 / 53,6%: B quasi raddoppia gli articoli tenuti prima del 2021 e lascia quasi invariato il 2025+, coerentemente con il recall recuperato nei periodi più vecchi.
- **Grafico a due pannelli.** Sopra, articoli per anno secondo la regola (azzurro chiaro) e secondo il filtro di rilevanza (blu). Sotto, la quota tenuta per anno, con la media del corpus tratteggiata; i due pannelli sono separati perché conteggi e quote hanno scale diverse, e la scala dei conteggi, dominata dal 2025-2026, nasconderebbe gli anni precedenti. Le barre grigie sono anni con meno di 10 articoli, dove un solo articolo sposta la quota di oltre 10 punti. Il 2026 è parziale (fino al 2026-09-25).
- **Etichette manuali contro decisione del modello.** Per le righe `y` e `n` l'accordo è perfetto per costruzione: il modello finale è addestrato proprio su quegli articoli e, con circa 400 feature su 175 esempi, li separa completamente. Non è una misura di qualità, che va presa dalla validazione incrociata (§5.4 e §5.7). La riga informativa è `p`, mai vista in addestramento: 16 tenute su 33, coerente con la loro natura di casi di confine.
- **Casi vicini alla soglia.** I 12 articoli con score più vicino a 0,245, nessuno annotato. Il gruppo è misto, come atteso: accanto ad articoli chiaramente rilevanti ("US immigration agents shot at people 16 times under Trump's crackdown") ce ne sono di apparentemente di passaggio ("Who are the key players in the resistance against Donald Trump?", tenuto per un margine minimo).

<!-- s5-relevance -->
### 5.6 Analisi di sensibilità: trattamento delle `p` e scelta del modello

Due scelte del §5.4 non sono ovvie e vanno giustificate con i numeri:

- trattamento delle `p`: escluse (`PARTIAL_AS = None`, scelta adottata), unite a `y` oppure unite a `n`;
- modello: A, A senza `has_ice_tag` (la feature che esiste quasi solo dal 2025, candidata a spiegare lo squilibrio di recall tra periodi), B.

Per ciascuna delle 9 combinazioni la cella ripete la procedura del §5.4 (5 fold, soglia a precisione ≥ `TARGET_PRECISION`, metriche pesate) e riporta il recall per periodo, gli articoli del corpus che il filtro terrebbe e quante delle 33 `p` verrebbero tenute. Il conteggio delle `p` è fuori campione solo nelle righe "p escluse"; nelle altre le `p` sono nel training e il numero riflette l'etichetta imposta. La cella non scrive file: è solo diagnostica.

Quando le `p` entrano nel training cambia anche la definizione di classe positiva su cui sono calcolate le metriche, quindi righe con trattamenti diversi non misurano esattamente la stessa cosa. Il confronto rigoroso è tra modelli a parità di trattamento.

In [ ]:
def sensitivity() -> pd.DataFrame:
    """Analisi di sensibilità: ripete la valutazione per 3 trattamenti delle p × 3 modelli.

    Per ogni combinazione riporta AP, precisione e recall pesati, recall per periodo e minimo,
    gli articoli del corpus che il filtro terrebbe e quante delle 33 p terrebbe. Non scrive file.
    """
    feature_sets = {
        "A": X_all,
        "A senza has_ice_tag": X_all.drop(columns=["has_ice_tag"]),
        "B": X_all.join(emb_all),
    }
    # id delle 33 p, per contare quante ne terrebbe ciascuna variante
    partial_ids = labeled.loc[labeled["label"] == "p", "id"]
    rows = []
    # 3 trattamenti delle p × 3 modelli = 9 combinazioni
    for p_name, partial_as in [("p escluse", None), ("p = y", 1), ("p = n", 0)]:
        data = build_train(partial_as)
        for m_name, X in feature_sets.items():
            summary, by_period = evaluate(m_name, X.loc[data["id"]], data)
            by_period = by_period.set_index("gruppo")
            # Variante addestrata su tutto il training e applicata al corpus intero: quanti articoli terrebbe
            score = make_model().fit(X.loc[data["id"]].values, data["y"].values).predict_proba(X.values)[:, 1]
            kept = pd.Series(score >= summary["soglia"], index=X.index)
            row = {
                "trattamento p": p_name, "modello": m_name, "n training": len(data),
                "AP": summary["avg precision"],
                "precision": by_period.loc["corpus (pesato)", "precision"],
                "recall": by_period.loc["corpus (pesato)", "recall"],
            }
            for period, _, _ in PERIODS:
                row[f"recall {period}"] = by_period.loc[period, "recall"]
            row["recall min"] = min(row[f"recall {period}"] for period, _, _ in PERIODS)
            row["articoli tenuti"] = int(kept.sum())
            row["p tenute (su 33)"] = int(kept.loc[partial_ids].sum())
            rows.append(row)
    return pd.DataFrame(rows).set_index(["trattamento p", "modello"])


if not ready:
    log.warning("§5.6 saltata: modello non addestrato (vedi §5.4)")
else:
    display(sensitivity().round(2))


**Come leggere la tabella.** Una riga per combinazione di trattamento delle `p` e modello. `AP`, `precision` e `recall` sono pesate sul corpus; le colonne `recall <periodo>` mostrano l'uniformità nel tempo, riassunta da `recall min`; `articoli tenuti` è la dimensione del corpus che ne risulterebbe. La precisione sta tra 0,90 e 0,93 in tutte le righe per costruzione della soglia, quindi il confronto si fa su recall e uniformità.

- Unire le `p` a una classe non aiuta: l'AP scende in tutte le varianti (0,94-0,96 contro 0,96-0,99), perché i casi di confine come etichette sono rumore.
- Con `p = y` il modello A perde gran parte del recall prima del 2021 (0,39-0,43): le `p` hanno poche menzioni, come le `n`, e le feature di conteggio e posizione non le distinguono. Con `p = n` il filtro diventa molto più severo (1.143 articoli) senza correggere lo squilibrio. Resta quindi `PARTIAL_AS = None`.
- Togliere `has_ice_tag` riduce lo squilibrio di A solo in parte (recall minimo da 0,57 a 0,65, al prezzo di un recall complessivo di 0,83): il tag non è l'unica causa. Anche le altre feature funzionano meglio sugli articoli recenti, che più spesso sono interamente dedicati all'agenzia.
- In ognuno dei tre trattamenti B ha il recall minimo più alto: è l'unica variante con recall alto e uniforme.

La riga "p escluse / A" dà 1.529 articoli contro i 1.528 del primo run: allora la soglia veniva arrotondata a 3 decimali prima di essere applicata, ora si usa il valore esatto (un articolo sul confine).

<!-- s5-relevance -->
### 5.7 Verifica con validazione incrociata annidata

Questa sezione controlla se le metriche del §5.4 (per B precisione 0,902, recall 0,959) sono gonfiate dal modo in cui viene scelta la soglia. Aggiunta il 2026-10-08; la cella non scrive file.

**1. Le previsioni out-of-fold.** Nel §5.4 i 175 articoli annotati sono divisi in 5 gruppi (fold) di circa 35. A turno un gruppo viene messo da parte, il modello si addestra sugli altri 140 e assegna una probabilità ai 35 esclusi. Dopo 5 giri ogni articolo ha una probabilità prodotta da un modello che non lo aveva visto: è il modo standard per stimare come il modello si comporta su articoli nuovi.

**2. Il problema: la soglia è scelta guardando le risposte.** Per trasformare le probabilità in decisioni serve una soglia. Nel §5.4 la soglia è la più bassa che dà precisione ≥ 0,90 sulle 175 previsioni out-of-fold, confrontate con le etichette vere, e precisione e recall vengono poi misurate su quelle stesse 175 previsioni. È come correggere un compito con le soluzioni in mano e decidere solo dopo dove mettere l'asticella. La precisione ≥ 0,90 è garantita per costruzione, non misurata, e la soglia si adatta anche alle particolarità casuali di questi 175 articoli. Il modello non ha visto le etichette dei 175 articoli, ma la scelta della soglia sì.

**3. La soluzione: un secondo livello di separazione.** La validazione incrociata annidata tiene da parte gli articoli anche durante la scelta della soglia:

```
175 articoli → 5 fold esterni da ~35
  per ogni fold esterno:
    ├─ 35 articoli di test: messi da parte, nessuno li guarda
    └─ 140 articoli di training:
         ├─ validazione incrociata interna a 5 fold sui soli 140 → probabilità out-of-fold dei 140
         ├─ soglia scelta su queste (precisione ≥ 0,90)       ← soglia scelta SENZA i 35
         ├─ modello addestrato sui 140
         └─ modello + soglia applicati ai 35 → decisioni tenuto/scartato
```

Dopo i 5 giri esterni ogni articolo ha una decisione presa da un modello che non lo ha visto e con una soglia scelta senza di lui. Le metriche calcolate su queste decisioni sono una misura vera, non garantita per costruzione. Ogni fold esterno sceglie la propria soglia, e anche questa variabilità entra nella stima. Modello, feature e regola per la soglia sono gli stessi del §5.4.

**4. Perché 5 semi.** Il seme decide come gli articoli vengono distribuiti tra i fold. Con 175 esempi la distribuzione conta: se un fold riceve per caso molti casi difficili, la stima si sposta di qualche punto. La procedura viene quindi ripetuta con 5 distribuzioni diverse (`NESTED_SEEDS`), e si riportano media e intervallo tra minimo e massimo.

**Come si interpreta.** Se le stime annidate sono vicine a quelle del §5.4, la scelta della soglia non gonfiava le metriche e i numeri del §5.4 si possono riportare così come sono. Se fossero molto più basse, le metriche del §5.4 andrebbero sostituite con quelle annidate. Il risultato è nella cella "come leggere" sotto le tabelle.

In [ ]:
# 5 suddivisioni diverse in fold: con 175 esempi la stima dipende da come cadono gli articoli
NESTED_SEEDS = [42, 1, 2, 3, 4]


def nested_cv(X: pd.DataFrame, data: pd.DataFrame, seed: int) -> np.ndarray:
    """Validazione incrociata annidata: restituisce una decisione 0/1 per ogni articolo di data.

    Per ogni fold esterno sceglie la soglia con una validazione incrociata interna sui soli articoli
    di training, addestra il modello sugli stessi articoli e lo applica a quelli esclusi con quella
    soglia. Nessun articolo contribuisce alla scelta della soglia con cui viene valutato.
    """
    Xv, y, w = X.loc[data["id"]].values, data["y"].values, data["w"].values
    pred = np.zeros(len(data), dtype=int)
    # Fold esterno: tr = ~140 articoli per addestrare e scegliere la soglia, te = ~35 da valutare
    for tr, te in StratifiedKFold(n_splits=5, shuffle=True, random_state=seed).split(Xv, y):
        # Soglia scelta con una CV interna sui soli articoli tr: quelli di te non entrano mai nella scelta
        inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
        proba_inner = cross_val_predict(make_model(), Xv[tr], y[tr], cv=inner, method="predict_proba")[:, 1]
        threshold = pick_threshold(y[tr], proba_inner, w[tr])
        pred[te] = make_model().fit(Xv[tr], y[tr]).predict_proba(Xv[te])[:, 1] >= threshold
    return pred


if not ready:
    log.warning("§5.7 saltata: modello non addestrato (vedi §5.4)")
else:
    y_tr, w_tr = train["y"].values, train["w"].values
    rows = []
    for seed in NESTED_SEEDS:
        for name, X in [(MODEL_A, X_all), (MODEL_B, X_all.join(emb_all))]:
            pred = nested_cv(X, train, seed)
            row = {"modello": name, "seed": seed,
                   "precision": precision_score(y_tr, pred, sample_weight=w_tr),
                   "recall": recall_score(y_tr, pred, sample_weight=w_tr)}
            for period, _, _ in PERIODS:
                m = (train["period"] == period).values
                row[f"precision {period}"] = precision_score(y_tr[m], pred[m], sample_weight=w_tr[m], zero_division=0)
                row[f"recall {period}"] = recall_score(y_tr[m], pred[m], sample_weight=w_tr[m], zero_division=0)
            rows.append(row)
    nested = pd.DataFrame(rows)

    # Confronto con la stima semplice del §5.4
    simple = per_period[per_period["gruppo"] == "corpus (pesato)"].set_index("modello")
    by_model = nested.groupby("modello")
    range_str = lambda s: f"{s.min():.3f}-{s.max():.3f}"
    display(pd.DataFrame({
        "precision CV semplice": simple["precision"],
        "precision annidata (media)": by_model["precision"].mean(),
        "precision annidata (min-max)": by_model["precision"].agg(range_str),
        "recall CV semplice": simple["recall"],
        "recall annidata (media)": by_model["recall"].mean(),
        "recall annidata (min-max)": by_model["recall"].agg(range_str),
    }).round(3))
    period_cols = [c for c in nested.columns if c.startswith(("precision ", "recall "))]
    display(by_model[period_cols].mean().T.round(3))

**Come leggere le tabelle.** Prima tabella: precisione e recall pesati sul corpus, con la validazione incrociata semplice del §5.4 accanto alla media e all'intervallo (minimo-massimo tra i 5 semi) della validazione annidata. Per B la stima annidata dà precisione 0,906 (0,897-0,914) e recall 0,947 (0,943-0,959), contro 0,902 e 0,959 della stima semplice; per A 0,919 e 0,880, contro 0,930 e 0,888. La distorsione dovuta alla scelta della soglia è al più di poco più di un punto e non cambia nessuna conclusione: la soglia scelta sulle previsioni out-of-fold è adeguata.

Seconda tabella: medie per periodo della stima annidata. Il quadro del §5.4 si conferma: recall di B 1,000 / 0,844 / 1,000 / 0,958, recall di A tra 0,59 e 0,69 prima del 2025. La precisione di B nel 2021-2024 è 0,793 (0,715 nella stima semplice): resta il periodo con la precisione più bassa.

## 6. Riepilogo: dai risultati dell'API al corpus finale

Schema dei passaggi che portano dai risultati delle due query al corpus usato nelle analisi successive (notebook 07, 08, 10), con il numero di articoli a ogni passaggio. I numeri sono letti dalle variabili delle sezioni precedenti, non scritti a mano.

In [ ]:
from matplotlib.patches import FancyBboxPatch

n_phrase, n_tag = (raw_df["query"] == "phrase").sum(), (raw_df["query"] == "tag").sum()
# Conteggi letti dalle variabili delle sezioni precedenti, non scritti a mano
stages = [
    ("API results", len(raw_df), f"phrase query {n_phrase:,}\ntag query {n_tag:,}"),
    ("Unique articles", len(df), f"{len(raw_df) - len(df):,} duplicates removed\n(articles returned by both queries)"),
    ("Rule filter", len(ice), "ICE tag OR agency full name\nin the text (§3)"),
    ("Relevance filter", int(out["is_ice_relevant"].sum()), f"model B, score ≥ {THRESHOLD:.3f}\n(§5)"),
]

fig, ax = plt.subplots(figsize=(12, 2.8))
ax.set_xlim(-0.01, 1.01)
ax.set_ylim(0, 1)
ax.axis("off")
# Quattro riquadri larghi 0,2, spazi uguali e un piccolo margine ai bordi
box_w, gap = 0.2, (1 - 4 * 0.2 - 0.02) / 3
for i, (title, n, note) in enumerate(stages):
    x0 = 0.01 + i * (box_w + gap)
    final = i == len(stages) - 1
    ax.add_patch(FancyBboxPatch((x0, 0.08), box_w, 0.84, boxstyle="round,pad=0.005,rounding_size=0.02",
                                facecolor="#2a78d6" if final else "#e8f0fb", edgecolor="#2a78d6", linewidth=1))
    color = "white" if final else "#1a1a1a"
    ax.text(x0 + box_w / 2, 0.78, title, ha="center", va="center", fontsize=10, color=color)
    ax.text(x0 + box_w / 2, 0.55, f"{n:,}", ha="center", va="center", fontsize=18, fontweight="bold", color=color)
    ax.text(x0 + box_w / 2, 0.25, note, ha="center", va="center", fontsize=8, color=color)
    if not final:
        ax.annotate("", xy=(x0 + box_w + gap - 0.004, 0.5), xytext=(x0 + box_w + 0.004, 0.5),
                    arrowprops={"arrowstyle": "->", "color": "#555555"})
ax.set_title("Guardian corpus construction (2003 to 2026-09-25)", fontsize=11)
plt.tight_layout()
fig.savefig(FIG_DIR / "guardian_pipeline.png", dpi=200, bbox_inches="tight")
plt.show()

**Come leggere lo schema.** Ogni riquadro è un passaggio della pipeline, con il numero di articoli che lo supera; l'ultimo, evidenziato, è il corpus usato dai notebook successivi. La perdita maggiore è nell'ultimo passaggio: il filtro a regola scarta solo 21 candidati, mentre il filtro di rilevanza esclude circa metà degli articoli che nominano l'agenzia. La figura è salvata in `docs/figures/guardian_pipeline.png` per la tesi.